# Yen carry trade unwind (August 2024)

This notebook reproduces the whole project from scratch: data download, cleaning,
modelling table, hypothesis tests (H1–H5) and robustness checks.

**How to run:** select the project's `.venv` kernel (see *Installation* in the README),
then *Run All*. Every step calls the code in `src/`; outputs are saved to
`data/`, `data_des/` and `results/` and shown below.

| Step | Module | Output |
|---|---|---|
| 1. Download | `src/data/*.py` | `data/*.parquet` |
| 2. Daily merge | `src/cleaning.py` | `data/merged_daily.parquet` |
| 3. Weekly table | `src/modelling_table.py` | `data/modelling_table.parquet` |
| 4. Data description | `src/data_description.py` | `data_des/` |
| 5. H1 skewness / kurtosis | `src/skewness_kurtosis.py` | printed |
| 6. H2–H4 quantile regressions | `src/quantile_regressions.py` | `results/quantile_coefs.*` |
| 7. H5 walk-forward | `src/walk_forward.py` | `results/walk_forward_*` |
| 8. Robustness | `src/robustness/*.py` | `results/robustness_*.csv` |
| 9. Tests | `tests/` | pytest report |

Design choices are documented in `docs/decisions.md`.

## 0. Setup

In [ ]:
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# Make `src` importable even without `pip install -e .`
ROOT = Path.cwd()
assert (ROOT / "src").is_dir(), "Open this notebook from the repository root"
sys.path.insert(0, str(ROOT))

from src import config

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)

# Set to True to re-download every source even if its file already exists
FORCE_DOWNLOAD = False

print("Analysis window:", config.START, "to", config.END)

## 1. Download the data

Sources: Yahoo Finance (FX, equities, VIX, Bitcoin, commodities), FRED (US/Japan
3-month rates), CFTC (yen futures positioning), BIS (yen credit outside Japan),
MoF Japan (FX interventions). Each module checks its file and adds a
`release_date` column so that later steps only use data that was public at the time.

A source is downloaded only if its file is missing (or `FORCE_DOWNLOAD = True`).
The CFTC download takes a few minutes.

In [ ]:
from src.data import bis, cftc, fred, mof, yahoo

downloads = {
    "yahoo": (yahoo.OUTPUT_FILE, lambda: yahoo.save_prices(yahoo.download_prices())),
    "fred":  (fred.OUTPUT_FILE,  lambda: fred.save_fred(fred.fetch_fred())),
    "cftc":  (cftc.OUTPUT_FILE,  lambda: cftc.save_cftc(cftc.fetch_cftc())),
    "bis":   (bis.OUTPUT_FILE,   lambda: bis.save_credit(bis.download_credit())),
    "mof":   (mof.OUTPUT_FILE,   lambda: mof.fetch_mof()),  # fetch_mof saves the file itself
}

for name, (path, download) in downloads.items():
    if FORCE_DOWNLOAD or not path.exists():
        print(f"Downloading {name}...")
        download()
    print(f"{name:6s} {len(pd.read_parquet(path)):6d} rows  ->  {path.relative_to(ROOT)}")

## 2. Daily merge

One row per USD/JPY trading day. Columns ending in `_known` only use a value from the
trading day after its release date (no look-ahead).

In [ ]:
from src import cleaning

daily = cleaning.build_daily(cleaning.load_sources())
cleaning.save_daily(daily)
print(f"{len(daily)} rows x {daily.shape[1]} columns")
daily.tail()

## 3. Weekly modelling table

Friday closes; targets are the 1-week and 4-week forward carry returns of a long USD / short JPY position
(spot move + rate differential); features use only data public on that Friday.

In [ ]:
from src import modelling_table

table = modelling_table.build_modelling_table()
table.to_parquet(config.DIRECTORY / "modelling_table.parquet", index=True)
print(f"{len(table)} weeks, {table.index[0].date()} to {table.index[-1].date()}")
table.describe().T

## 4. Data description

In [ ]:
runpy.run_module("src.data_description", run_name="__main__")

display(pd.read_csv(ROOT / "data_des" / "target_stats.csv", index_col=0))
for name in ["unwind_in_history", "2024_unwind", "hist_carry_4w"]:
    display(Image(ROOT / "data_des" / f"{name}.png"))

## 5. H1 — Negative skewness and fat left tails

Skewness and excess kurtosis of carry returns. The 1-week tests assume independent
weeks; the 4-week returns overlap, so their intervals come from a 12-week block bootstrap.

In [ ]:
from src import skewness_kurtosis

skewness_kurtosis.run()

## 6. H2–H4 — In-sample quantile regressions

4-week carry return on `spec_net_z` (H2, CFTC positioning), `carry_to_vol` (H3),
`credit_yoy` (H4, BIS yen credit) and `vix` (control), at the 5%, 50% and 95% quantiles and OLS.
Features scaled to mean 0 and std 1; 95% intervals from a 12-week block bootstrap
(500 draws, takes about a minute).

In [ ]:
from src import quantile_regressions

coefs = quantile_regressions.run()
display(coefs.pivot(index="variable", columns="model", values="coef"))
display(Image(quantile_regressions.RESULTS / "quantile_coefs.png"))

## 7. H5 — Walk-forward validation of the 5% quantile

Each week from July 2012 the model is refit on data known at that date only
(expanding window, 4-week embargo, features scaled on the training window) and its
5% quantile forecast is compared with the historical quantile and a volatility-scaled normal.

Scores: coverage (target 5%), Kupiec p-value, pinball loss (lower is better) and skill
versus the historical quantile (> 0 means better), with and without 2024.

In [ ]:
from src import walk_forward

scores = walk_forward.run()
display(scores)
display(Image(walk_forward.RESULTS / "walk_forward.png"))
display(Image(walk_forward.RESULTS / "walk_forward_2024.png"))

## 8. Robustness checks

Each variation changes one choice at a time (1-week target, 10% quantile, later start,
without 2024, without the 2025 rate break, without intervention weeks, 6-month BIS lag).

In [ ]:
from src.robustness import robustness_qr, robustness_wf

print("Quantile regressions:")
display(robustness_qr.run())
print("Walk-forward:")
display(robustness_wf.run())

## 9. Tests

Checks on the downloaded files, the merge, the modelling table and the walk-forward
(including the no-look-ahead test).

In [ ]:
import pytest

pytest.main(["-q", str(ROOT / "tests")])